In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')




In [ ]:
# Define paths
DIR = "/content/drive/MyDrive/NLP"
MODEL_DIR = f"{DIR}/ngram_model"
OUTPUT_DIR = f"{DIR}/Assignment6"
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load necessary libraries
import pickle
import random
import math
from collections import Counter, defaultdict

In [ ]:

def load_model(filename):
    with open(filename, "rb") as f:
        model = pickle.load(f)
    return model

In [ ]:
def load_sentences(filename):
    sentences = []
    with open(filename, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                if line.startswith("[") and line.endswith("]"):
                    sent = line[1:-1].split(", ")
                    sent = [w.strip("'\"") for w in sent]
                    sentences.append(sent)
                else:
                    sentences.append(line.split())
    return sentences



In [ ]:

bi_model=load_model(f'{MODEL_DIR}/final_2gram_counts.pkl')
uni_model=load_model(f'{MODEL_DIR}//final_1gram_counts.pkl')
tri_model=load_model(f'{MODEL_DIR}/final_3gram_counts.pkl')
quad_model=load_model(f'{MODEL_DIR}/final_4gram_counts.pkl')


In [ ]:
test_sentences = load_sentences(f"{MODEL_DIR}/test_sentences.csv")
print(f"Loaded {len(test_sentences)} test sentences.")
validation_sentences=load_sentences(f"{MODEL_DIR}/val_sentences.csv")
print(f"Loaded {len(validation_sentences)} val sentences.")

Loaded 1001 test sentences.
Loaded 1001 val sentences.


In [ ]:
%whos

Variable                   Type        Data/Info
------------------------------------------------
Counter                    type        <class 'collections.Counter'>
DIR                        str         /content/drive/MyDrive/NLP
MODEL_DIR                  str         /content/drive/MyDrive/NLP/ngram_model
OUTPUT_DIR                 str         /content/drive/MyDrive/NLP/Assignment6
bi_model                   dict        n=5143767
defaultdict                type        <class 'collections.defaultdict'>
drive                      module      <module 'google.colab.dri<...>s/google/colab/drive.py'>
generate_sentence_beam     function    <function generate_senten<...>e_beam at 0x7b2713fc82c0>
generate_sentence_greedy   function    <function generate_senten<...>greedy at 0x7b2713fc8360>
greedy_sentences           list        n=100
load_model                 function    <function load_model at 0x7b2820ed2520>
load_sentences             function    <function load_sentences at 0x7b2820ed176

# generate_sentence

In [ ]:
def generate_sentence_beam_quad(quad_model, beam_size=20, max_len=5):
    # Start with a random 3-word seed from the model
    start_seed = random.choice(list(quad_model.keys()))
    beams = [(list(start_seed[:3]), 0.0)]  # (sequence, log-prob)

    for _ in range(max_len):
        new_beams = []
        for seq, logp in beams:
            context = tuple(seq[-3:])
            # Get candidate next words that match context
            candidates = {k[-1]: v for k, v in quad_model.items() if k[:3] == context}
            if not candidates:
                new_beams.append((seq, logp))
                continue

            # Expand beams
            for word, count in candidates.items():
                new_seq = seq + [word]
                new_logp = logp + math.log(count + 1e-12)
                new_beams.append((new_seq, new_logp))

        # Keep only top beam_size sequences
        beams = sorted(new_beams, key=lambda x: x[1], reverse=True)[:beam_size]

    best_sentence = beams[0][0]
    return best_sentence

In [ ]:

def generate_sentence_greedy_quad(quad_model, max_len=20):
    # Choose a random quadrigram key as seed
    seed = random.choice(list(quad_model.keys()))
    sentence = list(seed[:3])  # first 3 words

    for _ in range(max_len):
        context = tuple(sentence[-3:])  # last 3 words
        candidates = {k[-1]: v for k,v in quad_model.items() if k[:3] == context}

        if not candidates:
            break  # stop if no matching candidates

        # pick next word with max count (greedy)
        next_word = max(candidates, key=candidates.get)
        sentence.append(next_word)

        if next_word == "</s>":
            break

    return sentence


In [ ]:
import re

def detokenize(tokens):
    """
    Convert list of word tokens into a readable sentence.
    Keeps punctuation attached to words.
    """
    sentence = " ".join(tokens)

    # remove space before punctuation
    sentence = re.sub(r'\s+([.,!?;:])', r'\1', sentence)

    return sentence


In [ ]:
num_sentences = 100
greedy_sentences = []

for i in range(num_sentences):
    sent = generate_sentence_greedy_quad(quad_model)
    greedy_sentences.append(sent)

    if (i+1) % 10 == 0:
        print(f" Generated {i+1}/{num_sentences} greedy sentences")
greedy_sentences_str = [detokenize(s) for s in greedy_sentences]

for i, s in enumerate(greedy_sentences_str[:10], 1):
    print(f"{i}: {s}")

print(greedy_sentences)

 Generated 10/100 greedy sentences
 Generated 20/100 greedy sentences
 Generated 30/100 greedy sentences
 Generated 40/100 greedy sentences
 Generated 50/100 greedy sentences
 Generated 60/100 greedy sentences
 Generated 70/100 greedy sentences
 Generated 80/100 greedy sentences
 Generated 90/100 greedy sentences
 Generated 100/100 greedy sentences
1: કરો n nEnglish Deutsch Espa ol fran ais italiano Portugu s Afrikaans Az rbaycan Dili Bisaya Bosanski Dansk Deutsch English Espa ol Estonia
2: title લોહી oldid 833726 " થી મેળવેલ n શ્રેણીઓ:,,,,,,,,,,,,,
3: તો આવું થાય એવું વિચારીને બંને ચલાવે રાખતાં હતાં. </s>
4: ChadhaJay BhattacharjeeJay JinaJayakrishnan NairJayalakshmi PJayant CharanJayaraman MahadevanJayasree SaranathanJijith Nadumuri RaviJithu AravamudanJoseph T NoonyJoydeep DattaJulianus PhilosophusJyotirgamayaJyotirmaya TripathyKal ChironKalavai VenkatKanimozhiKanu AgarwalKarthikeya TannaKartik MohanKaushik ChatterjeeKaveri MadhakKhatvaangaKhyati
5: વિના જ કાપી નાખી: ગમે તે ઘડીએ સર્

In [ ]:
import json
with open(f"{OUTPUT_DIR}/quadrigram_greedy.json","w") as f:
    json.dump(greedy_sentences, f, ensure_ascii=False, indent=4)

In [ ]:
num_sentences = 100
beam_sentences = []

for i in range(num_sentences):
    sent = generate_sentence_beam_quad(quad_model, beam_size=20)
    beam_sentences.append(sent)

    if (i+1) % 10 == 0:
        print(f"Generated {i+1}/{num_sentences} beam search sentences")

beam_sentences_str = [detokenize(s) for s in beam_sentences]

# Preview first 10
for i, s in enumerate(beam_sentences_str[:10], 1):
    print(f"{i}: {s}")





Generated 10/100 sentences
 Generated 20/100 sentences
 Generated 30/100 sentences
 Generated 40/100 sentences
Generated 50/100 sentences
Generated 60/100 sentences
Generated 70/100 sentences
Generated 80/100 sentences
Generated 90/100 sentences
Generated 100/100 sentences


In [ ]:
# Save to Drive


with open(f"{OUTPUT_DIR}/quadrigram_beam.json","w") as f:
    json.dump(beam_sentences, f, ensure_ascii=False, indent=4)